In [54]:
from dotenv import load_dotenv
load_dotenv()

True

In [55]:
from openai import OpenAI
openai_client = OpenAI()

In [56]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [57]:
question = "can I still join this course"

In [58]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [59]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [60]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
can I still join this course

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube 

In [61]:
# answer = llm(prompt)
# print(answer)

In [62]:
def build_prompt(question, search_results):
    return f"""
Your task is to answer questions from the course participants
based on the provided context.
Question: {question}
Context: {search_results}
"""

In [63]:
# answer = rag(question)
# print(answer)


In [64]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [65]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [66]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [67]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [68]:
# search_results = index.search(
#     question,
#     boost_dict={"question": 2.0, "section": 0.5},
#     filter_dict={"course": "llm-zoomcamp"},
#     num_results=3,
# )


In [74]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5,
    )


In [75]:
search_results = search(question)

In [76]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone proj

In [78]:
INSTRUCTION = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

"""

In [79]:
USER_Prompt = """
Question: {question}
context: {context}
"""

In [86]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [87]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: 

In [88]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_Prompt.format(
        question=question,
        context=context
    )

    return prompt.strip()

In [89]:
prompt = build_prompt(question, search_results)
print(prompt)

Question: can I still join this course
context: General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Does the course 